# Evaluation Framework

> Run and compare experiments across tasks and configurations


In [ ]:
#| default_exp evaluation


In [ ]:
#| exporti
from dataclasses import dataclass, field
from typing import Protocol, runtime_checkable
from collections import defaultdict
from pathlib import Path
from datetime import datetime
import json
import os
import logging
import tempfile
import shutil
import inspect
import time
import warnings


In [ ]:
#| hide
from nbdev.showdoc import *


In [ ]:
#| exporti
eval_logger = logging.getLogger("merceka_core.evaluation")


## Git Hash Tracking


In [ ]:
#| export
def get_git_hash() -> str:
    """Returns the current git commit hash or 'unknown' if not in a git repo."""
    try:
        from git import Repo
        from git.exc import InvalidGitRepositoryError
        repo = Repo(search_parent_directories=True)
        return repo.head.object.hexsha
    except Exception:
        return "unknown"


## Config Name Generation

Configs are dicts. We generate names from their contents, sorted alphabetically for determinism.


In [ ]:
#| export
def config_name(config: dict) -> str:
    """Generate a name from config contents.
    
    If a 'name' key is present, use it. Otherwise, auto-generate from
    other keys sorted alphabetically.
    
    Examples:
        >>> config_name({"rank": 64, "lr": 0.001})
        'lr_0.001_rank_64'
        >>> config_name({"name": "small", "rank": 64})
        'small'
        >>> config_name({})
        'default'
    """
    if "name" in config:
        return config["name"]
    
    # Filter to only primitive values that can be stringified cleanly
    parts = []
    for k, v in sorted(config.items()):
        if isinstance(v, (str, int, float, bool)):
            parts.append(f"{k}_{v}")
    
    if not parts:
        return "default"
    
    return "_".join(parts)


In [ ]:
# Test config_name
assert config_name({"rank": 64, "lr": 0.001}) == "lr_0.001_rank_64"
assert config_name({"name": "small", "rank": 64}) == "small"
assert config_name({}) == "default"
assert config_name({"b": 2, "a": 1}) == "a_1_b_2"  # Alphabetical
print("config_name tests passed!")


## Evaluation

A named metric computed by an evaluator.


In [ ]:
#| export
@dataclass
class Evaluation:
    """A named metric computed by an evaluator.
    
    Attributes:
        name: The metric name (e.g., 'success', 'perplexity', 'accuracy')
        value: The metric value (any type)
    """
    name: str
    value: object
    
    def to_dict(self) -> dict:
        """Convert to dictionary for serialization."""
        return {"name": self.name, "value": self.value}
    
    @classmethod
    def from_dict(cls, data: dict) -> "Evaluation":
        """Create from dictionary."""
        return cls(name=data["name"], value=data["value"])


## TaskResult

The outcome of a single run (one task × one config × one repetition).


In [ ]:
#| export
@dataclass
class TaskResult:
    """The outcome of a single run.
    
    Attributes:
        output: What the runner returned
        duration: Time taken in seconds
        task_name: Which task was run
        config_name: Which config was used
        evaluations: Metrics from evaluators
        error: Exception if the run failed, None otherwise
        trace_id: External tracing ID (optional)
    """
    output: object
    duration: float
    task_name: str
    config_name: str
    evaluations: list[Evaluation] = field(default_factory=list)
    error: Exception | None = None
    trace_id: str | None = None
    
    def get_evaluation(self, name: str) -> Evaluation | None:
        """Get a specific evaluation by name."""
        return next((e for e in self.evaluations if e.name == name), None)
    
    @property
    def success(self) -> bool | None:
        """Convenience: get the 'success' evaluation if it exists."""
        eval = self.get_evaluation("success")
        return eval.value if eval else None
    
    def to_dict(self) -> dict:
        """Convert to dictionary for serialization."""
        return {
            "output": self.output,
            "duration": self.duration,
            "task_name": self.task_name,
            "config_name": self.config_name,
            "evaluations": [e.to_dict() for e in self.evaluations],
            "error": str(self.error) if self.error else None,
            "error_type": type(self.error).__name__ if self.error else None,
            "trace_id": self.trace_id,
        }
    
    @classmethod
    def from_dict(cls, data: dict) -> "TaskResult":
        """Create from dictionary."""
        # Reconstruct error as a simple Exception with the message
        error = None
        if data.get("error"):
            error_type = data.get("error_type", "Exception")
            error = Exception(f"[{error_type}] {data['error']}")
        
        return cls(
            output=data["output"],
            duration=data["duration"],
            task_name=data["task_name"],
            config_name=data["config_name"],
            evaluations=[Evaluation.from_dict(e) for e in data.get("evaluations", [])],
            error=error,
            trace_id=data.get("trace_id"),
        )


## ExperimentResults

A collection of TaskResults with aggregation and slicing capabilities.


In [ ]:
#| export
@dataclass
class ExperimentResults:
    """Collection of TaskResults with aggregation and slicing.
    
    Supports:
    - Slicing by task or config: results["task_name"] or results["config_name"]
    - Iteration: for r in results
    - Indexing: results[0]
    - Combining: results1 | results2
    - Persistence: results.save(path), ExperimentResults.load(path)
    
    Attributes:
        results: List of TaskResult objects
        experiment_name: Name of the experiment
        description: Human-readable description
        git_hash: Git commit hash for reproducibility
        save_path: Where results were saved (if saved)
    """
    results: list[TaskResult]
    experiment_name: str
    description: str | None = None
    git_hash: str = field(default_factory=get_git_hash)
    save_path: str | None = None
    
    # ─────────────────────────────────────────────────────────────
    # Iteration and Indexing
    # ─────────────────────────────────────────────────────────────
    
    def __iter__(self):
        return iter(self.results)
    
    def __len__(self):
        return len(self.results)
    
    def __getitem__(self, key):
        """Index by integer or slice by task/config name."""
        if isinstance(key, int):
            return self.results[key]
        elif isinstance(key, str):
            return self._slice_by_name(key)
        else:
            raise TypeError(f"Invalid key type: {type(key)}")
    
    # ─────────────────────────────────────────────────────────────
    # Slicing
    # ─────────────────────────────────────────────────────────────
    
    def _slice_by_name(self, key: str) -> "ExperimentResults":
        """Filter results by task or config name."""
        is_task = any(r.task_name == key for r in self.results)
        is_config = any(r.config_name == key for r in self.results)
        
        if is_task and is_config:
            raise KeyError(
                f"'{key}' is ambiguous — matches both task and config.\n"
                f"Use by_task('{key}') or by_config('{key}')."
            )
        elif is_task:
            return self.by_task(key)
        elif is_config:
            return self.by_config(key)
        else:
            task_names = sorted(set(r.task_name for r in self.results))
            config_names = sorted(set(r.config_name for r in self.results))
            raise KeyError(
                f"No task or config named '{key}'.\n"
                f"Available tasks: {task_names}\n"
                f"Available configs: {config_names}"
            )
    
    def by_task(self, name: str) -> "ExperimentResults":
        """Filter to results for a specific task."""
        filtered = [r for r in self.results if r.task_name == name]
        return ExperimentResults(
            results=filtered,
            experiment_name=self.experiment_name,
            description=self.description,
            git_hash=self.git_hash,
            save_path=self.save_path,
        )
    
    def by_config(self, name: str) -> "ExperimentResults":
        """Filter to results for a specific config."""
        filtered = [r for r in self.results if r.config_name == name]
        return ExperimentResults(
            results=filtered,
            experiment_name=self.experiment_name,
            description=self.description,
            git_hash=self.git_hash,
            save_path=self.save_path,
        )
    
    @property
    def task_names(self) -> set[str]:
        """All unique task names in results."""
        return set(r.task_name for r in self.results)
    
    @property
    def config_names(self) -> set[str]:
        """All unique config names in results."""
        return set(r.config_name for r in self.results)
    
    # ─────────────────────────────────────────────────────────────
    # Aggregation
    # ─────────────────────────────────────────────────────────────
    
    @property
    def total_duration(self) -> float:
        """Sum of all run durations."""
        return sum(r.duration for r in self.results)
    
    @property
    def average_duration(self) -> float:
        """Average run duration."""
        if not self.results:
            return 0.0
        return self.total_duration / len(self.results)
    
    def get_evaluation_values(self, name: str) -> list:
        """Get all values for a named evaluation across results."""
        values = []
        for r in self.results:
            eval = r.get_evaluation(name)
            if eval:
                values.append(eval.value)
        return values
    
    @property
    def success_rate(self) -> float | None:
        """Success percentage (0-100) if 'success' evaluation exists.
        
        Only counts boolean True/False values. Non-boolean values are ignored.
        """
        successes = self.get_evaluation_values("success")
        if not successes:
            return None
        # Only count boolean values to avoid type errors
        bool_successes = [s for s in successes if isinstance(s, bool)]
        if not bool_successes:
            return None
        return sum(bool_successes) / len(bool_successes) * 100
    
    @property
    def failures(self) -> "ExperimentResults":
        """Results where error is not None."""
        filtered = [r for r in self.results if r.error is not None]
        return ExperimentResults(
            results=filtered,
            experiment_name=self.experiment_name,
            description=self.description,
            git_hash=self.git_hash,
            save_path=self.save_path,
        )
    
    @property
    def successes(self) -> "ExperimentResults":
        """Results where error is None."""
        filtered = [r for r in self.results if r.error is None]
        return ExperimentResults(
            results=filtered,
            experiment_name=self.experiment_name,
            description=self.description,
            git_hash=self.git_hash,
            save_path=self.save_path,
        )
    
    # ─────────────────────────────────────────────────────────────
    # Combining
    # ─────────────────────────────────────────────────────────────
    
    def __or__(self, other: "ExperimentResults | None") -> "ExperimentResults":
        """Combine two ExperimentResults."""
        if other is None:
            return self
        return ExperimentResults(
            results=self.results + other.results,
            experiment_name=self.experiment_name,
            description=self.description,
            git_hash=self.git_hash,
            save_path=self.save_path,
        )
    
    # ─────────────────────────────────────────────────────────────
    # Persistence
    # ─────────────────────────────────────────────────────────────
    
    def save(self, path: str):
        """Save results to disk.
        
        Creates:
        - experiment_results.json: Full serialized data
        - result_summary.txt: Human-readable summary
        """
        path = Path(path)
        path.mkdir(parents=True, exist_ok=True)
        
        # Serialize to dict
        data = {
            "experiment_name": self.experiment_name,
            "description": self.description,
            "git_hash": self.git_hash,
            "results": [r.to_dict() for r in self.results],
        }
        
        # Atomic write: write to temp file, then rename
        results_file = path / "experiment_results.json"
        with tempfile.NamedTemporaryFile(mode='w', dir=path, suffix='.json', delete=False) as f:
            json.dump(data, f, indent=2, default=str)
            temp_path = f.name
        shutil.move(temp_path, results_file)
        
        # Save summary
        summary_file = path / "result_summary.txt"
        summary_file.write_text(self.summary())
        
        self.save_path = str(path)
    
    @classmethod
    def load(cls, path: str) -> "ExperimentResults":
        """Load results from disk."""
        path = Path(path)
        
        # Handle both directory and file paths
        if path.is_dir():
            results_file = path / "experiment_results.json"
        else:
            results_file = path
            path = path.parent
        
        if not results_file.exists():
            raise FileNotFoundError(f"No results file found at {results_file}")
        
        with open(results_file) as f:
            data = json.load(f)
        
        return cls(
            results=[TaskResult.from_dict(r) for r in data["results"]],
            experiment_name=data["experiment_name"],
            description=data.get("description"),
            git_hash=data.get("git_hash", "unknown"),
            save_path=str(path),
        )
    
    # ─────────────────────────────────────────────────────────────
    # Display
    # ─────────────────────────────────────────────────────────────
    
    def summary(self) -> str:
        """Generate a human-readable summary."""
        lines = [
            f"{self.experiment_name} Results",
            f"Description: {self.description or 'N/A'}",
            f"Git hash: {self.git_hash}",
            f"Total runs: {len(self.results)}",
            f"Total duration: {self.total_duration:.2f} seconds",
            "",
        ]
        
        # Success rate if available
        if self.success_rate is not None:
            lines.append(f"Success rate: {self.success_rate:.1f}%")
            lines.append("")
        
        # By task
        if len(self.task_names) > 1:
            lines.append("By Task:")
            for task in sorted(self.task_names):
                subset = self.by_task(task)
                rate = subset.success_rate
                rate_str = f"{rate:.1f}%" if rate is not None else "N/A"
                lines.append(f"  {task}: {len(subset)} runs, {rate_str}")
            lines.append("")
        
        # By config
        if len(self.config_names) > 1:
            lines.append("By Config:")
            for config in sorted(self.config_names):
                subset = self.by_config(config)
                rate = subset.success_rate
                rate_str = f"{rate:.1f}%" if rate is not None else "N/A"
                lines.append(f"  {config}: {len(subset)} runs, {rate_str}")
            lines.append("")
        
        # Failures
        num_failures = len(self.failures)
        if num_failures > 0:
            lines.append(f"Failures: {num_failures}")
        
        return "\n".join(lines)
    
    def __repr__(self) -> str:
        task_count = len(self.task_names)
        config_count = len(self.config_names)
        return (
            f"ExperimentResults('{self.experiment_name}', "
            f"{len(self.results)} runs, "
            f"{task_count} tasks, "
            f"{config_count} configs)"
        )


In [ ]:
# Test ExperimentResults
results = ExperimentResults(
    results=[
        TaskResult(output="out1", duration=1.0, task_name="task_a", config_name="config_1",
                   evaluations=[Evaluation("success", True)]),
        TaskResult(output="out2", duration=2.0, task_name="task_a", config_name="config_2",
                   evaluations=[Evaluation("success", False)]),
        TaskResult(output="out3", duration=1.5, task_name="task_b", config_name="config_1",
                   evaluations=[Evaluation("success", True)]),
    ],
    experiment_name="test_experiment",
)

# Test iteration
assert len(results) == 3
assert results[0].output == "out1"

# Test slicing
assert len(results["task_a"]) == 2
assert len(results["config_1"]) == 2

# Test aggregation
assert results.total_duration == 4.5
assert abs(results.success_rate - 66.67) < 0.1

print(results)
print()
print(results.summary())


In [ ]:
# Test save/load round-trip
import tempfile as tf
with tf.TemporaryDirectory() as tmpdir:
    results.save(tmpdir)
    loaded = ExperimentResults.load(tmpdir)
    assert len(loaded) == len(results)
    assert loaded.experiment_name == results.experiment_name
    assert loaded[0].output == results[0].output
    print("Save/load round-trip passed!")


## Protocols

Minimal interfaces for tasks and evaluators.


In [ ]:
#| export
@runtime_checkable
class Task(Protocol):
    """Minimal task protocol - just needs a name.
    
    Any object with a `name` attribute satisfies this protocol.
    You can add any additional fields you need.
    
    Examples:
        @dataclass
        class MyTask:
            name: str
            prompt: str
            expected_output: str
    """
    name: str


In [ ]:
#| export
@runtime_checkable
class Evaluator(Protocol):
    """Protocol for evaluators that compute metrics from run outputs.
    
    Evaluators are called after each run to compute metrics.
    They must have a name (for display) and an evaluate method.
    
    The evaluate method receives:
    - output: what the runner returned
    - task: the task that was run (if provided)
    - config: the config that was used (if provided)
    
    Returns a list of Evaluation objects.
    
    Example:
        class SuccessEvaluator:
            name = "success_checker"
            
            def evaluate(self, output, task=None, config=None) -> list[Evaluation]:
                success = output.get("completed", False)
                return [Evaluation("success", success)]
    """
    name: str
    
    def evaluate(self, output, task=None, config=None) -> list[Evaluation]:
        ...


## Experiment Class

The central abstraction for running experiments. Handles:
- Running functions across tasks × configs × repetitions
- Automatic calling convention detection
- Evaluator execution
- Error handling (continue on failure)
- Saving results


In [ ]:
#| export
def _detect_calling_convention(func, has_tasks: bool, has_configs: bool) -> str:
    """Detect how to call the runner function based on its signature.
    
    Returns one of:
    - 'none': func()
    - 'task': func(task)
    - 'config': func(**config)
    - 'both': func(task, **config)
    """
    sig = inspect.signature(func)
    params = list(sig.parameters.keys())
    
    # Check for **kwargs
    has_kwargs = any(
        p.kind == inspect.Parameter.VAR_KEYWORD 
        for p in sig.parameters.values()
    )
    
    if not params:
        return 'none'
    
    # If first param exists and we have tasks, assume it's for tasks
    first_param = params[0] if params else None
    accepts_task = first_param is not None and has_tasks
    accepts_config = has_kwargs or (has_configs and len(params) > (1 if accepts_task else 0))
    
    if accepts_task and accepts_config:
        return 'both'
    elif accepts_task:
        return 'task'
    elif accepts_config:
        return 'config'
    else:
        return 'none'


In [ ]:
#| export
def _generate_save_path(name: str) -> str:
    """Generate a unique save path for experiment results."""
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    return f"./experiment_results/{name}_{timestamp}"


In [ ]:
#| export
def _check_duplicate_config_names(configs: list[dict]):
    """Raise error if any configs have the same name."""
    names = [config_name(c) for c in configs]
    seen = set()
    for n in names:
        if n in seen:
            raise ValueError(
                f"Duplicate config name: '{n}'.\n"
                f"All config names: {names}\n"
                f"Use 'name' key in config to provide unique names."
            )
        seen.add(n)


In [ ]:
#| export
def _check_duplicate_evaluation_names(evaluations: list[Evaluation]):
    """Raise error if evaluations have duplicate names."""
    names = [e.name for e in evaluations]
    seen = set()
    for n in names:
        if n in seen:
            raise ValueError(
                f"Duplicate evaluation name: '{n}'.\n"
                f"All evaluation names: {names}\n"
                f"Ensure each evaluator produces uniquely named evaluations."
            )
        seen.add(n)


In [ ]:
#| export
class Experiment:
    """Run experiments across tasks × configs × repetitions.
    
    The simplest usage:
        exp = Experiment(name="quick_test", run=my_function)
        results = exp.run()
    
    Full usage:
        exp = Experiment(
            name="grid_search",
            run=train_model,
            tasks=[task1, task2],
            configs=[{"lr": 0.1}, {"lr": 0.01}],
            repetitions=3,
            evaluators=[accuracy_eval, loss_eval],
            description="Comparing learning rates"
        )
        results = exp.run()
    
    Args:
        name: Required. Experiment name for saving and display.
        run: Required. Function to run for each (task, config) pair.
        tasks: Optional list of tasks (anything with a .name attribute).
        configs: Optional list of config dicts.
        repetitions: Number of times to repeat each (task, config) pair.
        evaluators: Optional list of Evaluator objects.
        description: Optional human-readable description.
        save: Whether to save results (default True).
        save_path: Custom save path (auto-generated if not provided).
    """
    
    def __init__(
        self,
        name: str,
        run,
        tasks: list | None = None,
        configs: list[dict] | None = None,
        repetitions: int = 1,
        evaluators: list | None = None,
        description: str | None = None,
        save: bool = True,
        save_path: str | None = None,
    ):
        self.name = name
        self.run_func = run
        self.tasks = tasks
        self.configs = configs
        self.repetitions = repetitions
        self.evaluators = evaluators or []
        self.description = description
        self.save = save
        self.save_path = save_path or _generate_save_path(name) if save else None
        
        # Validate configs upfront
        if self.configs:
            _check_duplicate_config_names(self.configs)
    
    def run(self) -> ExperimentResults:
        """Execute the experiment and return results."""
        results: list[TaskResult] = []
        
        # Determine what we're iterating over
        tasks = self.tasks or [None]
        configs = self.configs or [{}]
        has_tasks = self.tasks is not None
        has_configs = self.configs is not None
        
        # Detect calling convention
        convention = _detect_calling_convention(self.run_func, has_tasks, has_configs)
        
        # Main loop: tasks × configs × repetitions
        total_runs = len(tasks) * len(configs) * self.repetitions
        run_count = 0
        
        for task in tasks:
            task_name = task.name if task else "default"
            
            for config in configs:
                cfg_name = config_name(config)
                
                for rep in range(self.repetitions):
                    run_count += 1
                    eval_logger.info(f"Run {run_count}/{total_runs}: {task_name} x {cfg_name} (rep {rep+1})")
                    
                    result = self._run_single(task, config, task_name, cfg_name, convention)
                    results.append(result)
                    
                    # Save intermediate results
                    if self.save and self.save_path:
                        exp_results = ExperimentResults(
                            results=results,
                            experiment_name=self.name,
                            description=self.description,
                        )
                        exp_results.save(self.save_path)
        
        final_results = ExperimentResults(
            results=results,
            experiment_name=self.name,
            description=self.description,
            save_path=self.save_path,
        )
        
        return final_results
    
    def _run_single(self, task, config: dict, task_name: str, cfg_name: str, convention: str) -> TaskResult:
        """Run a single (task, config) combination."""
        start_time = time.time()
        output = None
        error = None
        
        try:
            # Call based on convention
            if convention == 'none':
                output = self.run_func()
            elif convention == 'task':
                output = self.run_func(task)
            elif convention == 'config':
                # Filter out 'name' key from config before passing
                run_config = {k: v for k, v in config.items() if k != 'name'}
                output = self.run_func(**run_config)
            elif convention == 'both':
                run_config = {k: v for k, v in config.items() if k != 'name'}
                output = self.run_func(task, **run_config)
        except Exception as e:
            error = e
            warnings.warn(f"Run failed: {task_name} x {cfg_name}: {e}")
        
        duration = time.time() - start_time
        
        # Run evaluators (only if run succeeded - error is None)
        # Note: output can legitimately be None, so we check error instead
        evaluations = []
        if error is None:
            for evaluator in self.evaluators:
                try:
                    evals = evaluator.evaluate(output, task=task, config=config)
                    evaluations.extend(evals)
                except Exception as e:
                    warnings.warn(f"Evaluator '{evaluator.name}' failed: {e}")
            
            # Check for duplicate evaluation names
            if evaluations:
                _check_duplicate_evaluation_names(evaluations)
        
        return TaskResult(
            output=output,
            duration=duration,
            task_name=task_name,
            config_name=cfg_name,
            evaluations=evaluations,
            error=error,
        )


In [ ]:
#| export
def run_experiment(
    name: str,
    run,
    tasks: list | None = None,
    configs: list[dict] | None = None,
    repetitions: int = 1,
    evaluators: list | None = None,
    description: str | None = None,
    save: bool = True,
    save_path: str | None = None,
) -> ExperimentResults:
    """Convenience function to run an experiment.
    
    This is a thin wrapper around Experiment for quick one-off experiments.
    
    Examples:
        # Simplest case
        results = run_experiment(name="test", run=my_func)
        
        # With configs
        results = run_experiment(
            name="compare_lr",
            run=train,
            configs=[{"lr": 0.1}, {"lr": 0.01}]
        )
    
    See Experiment class for full documentation.
    """
    exp = Experiment(
        name=name,
        run=run,
        tasks=tasks,
        configs=configs,
        repetitions=repetitions,
        evaluators=evaluators,
        description=description,
        save=save,
        save_path=save_path,
    )
    return exp.run()


## Integration Tests


In [ ]:
# Test 1: Simplest case - just a function
def simple_func():
    return {"value": 42}

results = run_experiment(name="simple_test", run=simple_func, save=False)
assert len(results) == 1
assert results[0].output == {"value": 42}
print("Test 1 passed: Simple function")


In [ ]:
# Test 2: With configs
def configurable_func(lr, batch_size=32):
    return {"lr": lr, "batch_size": batch_size}

results = run_experiment(
    name="config_test",
    run=configurable_func,
    configs=[
        {"name": "small", "lr": 0.1, "batch_size": 16},
        {"name": "large", "lr": 0.01, "batch_size": 64},
    ],
    save=False
)
assert len(results) == 2
assert results["small"][0].output["lr"] == 0.1
assert results["large"][0].output["batch_size"] == 64
print("Test 2 passed: With configs")


In [ ]:
# Test 3: With tasks
@dataclass
class SimpleTask:
    name: str
    input_text: str

def task_runner(task):
    return {"processed": task.input_text.upper()}

tasks = [
    SimpleTask("task_a", "hello"),
    SimpleTask("task_b", "world"),
]

results = run_experiment(name="task_test", run=task_runner, tasks=tasks, save=False)
assert len(results) == 2
assert results["task_a"][0].output["processed"] == "HELLO"
assert results["task_b"][0].output["processed"] == "WORLD"
print("Test 3 passed: With tasks")


In [ ]:
# Test 4: With evaluators
class SuccessEvaluator:
    name = "success_check"
    
    def evaluate(self, output, task=None, config=None):
        return [Evaluation("success", output.get("value", 0) > 0)]

def value_func():
    return {"value": 10}

results = run_experiment(
    name="eval_test",
    run=value_func,
    evaluators=[SuccessEvaluator()],
    save=False
)
assert results[0].success == True
assert results.success_rate == 100.0
print("Test 4 passed: With evaluators")


In [ ]:
# Test 5: Full grid search (tasks × configs × repetitions)
def full_func(task, lr=0.1):
    return {"task": task.name, "lr": lr}

results = run_experiment(
    name="grid_test",
    run=full_func,
    tasks=tasks,  # 2 tasks
    configs=[{"name": "fast", "lr": 0.1}, {"name": "slow", "lr": 0.01}],  # 2 configs
    repetitions=2,
    save=False
)
assert len(results) == 8  # 2 * 2 * 2
print("Test 5 passed: Full grid search")


In [ ]:
# Test 6: Error handling (continues on failure)
call_count = 0
def flaky_func():
    global call_count
    call_count += 1
    if call_count == 2:
        raise ValueError("Intentional failure")
    return {"run": call_count}

call_count = 0
results = run_experiment(
    name="flaky_test",
    run=flaky_func,
    repetitions=3,
    save=False
)
assert len(results) == 3
assert len(results.failures) == 1
assert len(results.successes) == 2
print("Test 6 passed: Error handling")


## Analysis Utilities

Simple functions for working with results. These are not part of the core export, just utilities.


In [ ]:
#| export
def to_dataframe(results: ExperimentResults):
    """Convert ExperimentResults to a pandas DataFrame.
    
    Requires pandas to be installed.
    """
    import pandas as pd
    
    rows = []
    for r in results:
        row = {
            "task_name": r.task_name,
            "config_name": r.config_name,
            "duration": r.duration,
            "error": str(r.error) if r.error else None,
        }
        # Add all evaluations as columns
        for e in r.evaluations:
            row[e.name] = e.value
        rows.append(row)
    
    return pd.DataFrame(rows)
